In [138]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any, ClassVar

In [139]:
@dataclass
class Performer:
    """A performer in the Yes-And game."""
    
    # Instance fields (dataclass will auto-generate __init__)
    name: str
    role: str  # "host" or "player"
    client_config: Dict[str, str]  # Store config instead of client
    model: str
    system_prompt: str = ""
    temperature: float = 0.7
    max_tokens: int = 500

    client: OpenAI = field(init=False, default=None)
    
    def __post_init__(self):
        """Create client from stored config."""
        if self.client_config.get('base_url'):
            self.client = OpenAI(api_key=self.client_config['api_key'], base_url=self.client_config['base_url'])
        else:
            self.client = OpenAI(api_key=self.client_config.get('api_key'))
    
    # Shared across ALL Performer instances
    full_history: ClassVar[List[Dict[str, str]]] = []

    def set_system_prompt(self, system_prompt: str) -> None:
        """Reset or update the system prompt."""
        self.system_prompt = system_prompt

    def set_starting_message(self, starting_message: str) -> None:
        """Return the starting message for the performer."""
        msg = f"{self.name} says: {starting_message}"
        Performer.full_history = [{"role": "assistant", "content": msg}]

    def _chat(self, extra_user_msg: Optional[str] = None) -> str:
        """Internal helper to call the model with full history and append response."""
        history: List[Dict[str, str]] = [{"role": "system", "content": self.system_prompt}]
        history.extend(Performer.full_history)
        if extra_user_msg:
            history.append({"role": "user", "content": extra_user_msg})

        resp = self.client.chat.completions.create(
            model=self.model,
            messages=history,
            temperature=self.temperature,
            max_tokens=self.max_tokens
        )
        content = resp.choices[0].message.content

        # Label with performer identity (so others know who said it)
        # Only add label if it's not already present
        if content.startswith(f"{self.name} says:"):
            labeled_content = content
        else:
            labeled_content = f"{self.name} says: {content}"

        # Save assistant reply into shared history
        Performer.full_history.append({"role": "assistant", "content": labeled_content})
        return labeled_content
    
    def user_interaction(self, user_message: str) -> str:
        """Continue conversation with full history (host + user)."""
        Performer.full_history.append({"role": "user", "content": user_message})
        return self._chat()

    def decide(self) -> str:
        """
        Host decides if the game should continue or end.
        By default, let the model make the decision based on the history.
        """
        assert self.role == "host", "Only the host should decide."
        decision_prompt = (
            "Based on the scene so far, decide whether to START, CONTINUE or END GAME. "
            "Reply with only 'start', 'continue' or 'end'."
        )
        decision = self._chat(extra_user_msg=decision_prompt).lower()

        if "end" in decision:
            return "end"
        return "continue" 
    
    def speak(self) -> str:
        """
        Player speaks by building on the last assistant message in history.
        Example: Wayne uses Ryan's last line as context for his next move.
        """
        # Find the last assistant message (could be host or another player)
        last_line: Optional[str] = None
        for msg in reversed(Performer.full_history):
            if msg["role"] == "assistant":
                last_line = msg["content"]
                break

        if last_line is None:
            # If no assistant messages yet, just let the model go
            return self._chat()

        # Add a user prompt to encourage Yes-And on last line
        extra_prompt = f"Continue the scene by building on this: {Performer.full_history}"
        return self._chat(extra_user_msg=extra_prompt)

    @classmethod
    def get_full_history(cls) -> List[Dict[str, str]]:
        """Return the full conversation so far."""
        return cls.full_history

    
    @classmethod
    def clear_full_history(cls) -> None:
        """Reset the shared history."""
        cls.full_history = []

In [140]:
load_dotenv(override=True)

# create client configurations for different models
anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

# instantiate performers with client configs instead of client objects
host = Performer(
    name="Drew",
    role="host", 
    client_config={"api_key": os.getenv('OPENAI_API_KEY')}, 
    model="gpt-4o-mini")
    
performer1 = Performer(
    name="Ryan", 
    role="player", 
    client_config={"api_key": os.getenv('GOOGLE_API_KEY'), "base_url": gemini_url}, 
    model="gemini-2.5-flash")

performer2 = Performer(
    name="Wayne", 
    role="player", 
    client_config={"api_key": os.getenv('ANTHROPIC_API_KEY'), 
    "base_url": anthropic_url}, 
    model="claude-3-5-haiku-latest")

In [141]:
host_system_prompt = f"""You are {host.name}, the host of a multi-agent “Yes, And” improv game.You do not play the game yourself—you only guide it, 
moderate it, and make decisions.
Responsibilities
Solicit Scenario
Ask the user (audience) for a fun scenario to start the game.If unclear/inappropriate, ask them to rephrase once.
Frame the Scene
Convert the scenario into a structured Scene Brief (setting, tone, and constraints).
Broadcast this Scene Brief as instructions to the players. The instructions for {performer1.name} and {performer2.name}
should not be more than 3 sentences. You should not make up names for performers. It is there story to tell. Just 
give them the scene brief and let them play.
To start the game, say [HOST DECISION: Start Game] to start the game loop.
The Game Loop
Alternate turns between {performer1.name} (Player 1) and {performer2.name} (Player 2).
After each pair of turns, decide whether to continue or end.
If ending, wrap up with a closing message to the user.
Decision Making
Say [HOST DECISION: Start Game] to start the game. Only say this once to start the game loop
Say [HOST DECISION: Continue Game] to keep the game going.
Say [HOST DECISION: End Game] to stop the game."""

performer1_system_prompt = f"""You are {performer1.name}, Player 1 in the improv game Yes, And.
You play inside the scene brief that {host.name} (the host) provides.
Responsibilities
Always accept what has been established (the “Yes”).Always add something new that pushes the story forward (the “And”).
Stay within the tone, rules, and constraints that {host.name} defines.
Write 2–3 sentences per turn (unless {host.name} specifies otherwise).
Never act as {host.name} or {performer2.name} — only roleplay your own turn."""

performer2_system_prompt = f"""You are {performer2.name}, Player 2 in the improv game Yes, And.
You play inside the scene brief that {host.name} (the host) provides.
Responsibilities
Always accept what has been established (the “Yes”).Always add something new that pushes the story forward (the “And”).
Stay within the tone, rules, and constraints that {host.name} defines.
Write 2–3 sentences per turn (unless {host.name} specifies otherwise).
Never act as {host.name} or {performer1.name} — only roleplay your own turn."""

In [142]:
host.set_system_prompt(host_system_prompt)
performer1.set_system_prompt(performer1_system_prompt)
performer2.set_system_prompt(performer2_system_prompt)

In [143]:
print(host.name)
print(performer1.name)
print(performer2.name)

Drew
Ryan
Wayne


In [144]:
host.set_starting_message(f'''Hello, I am {host.name} and I am the host of the Yes-And game. 
    Lets get started!''')

print(host.user_interaction("What do I do?."))




Drew says: Please provide a fun scenario to start the game. It could be anything—like a situation, a location, or a character—just make it creative!


In [145]:
print(host.user_interaction("How about a pair of dogs that can fly and fight crime."))


Drew says: Great scenario! Here’s the Scene Brief:

**Setting:** A vibrant city skyline with a mix of tall buildings and parks.  
**Tone:** Light-hearted and adventurous.  
**Constraints:** The flying dogs must communicate with each other using dog-like sounds and body language, and they should encounter a mischievous villain trying to steal dog treats.

Ryan and Wayne, your scene begins with you as these flying crime-fighting dogs! 

[HOST DECISION: Start Game]


In [146]:
print(performer1.speak())

Ryan says: (As Barker, the flying dog)

Woof! Woof! My name is Barker, and I'm soaring high above the city, my tail wagging a steady rhythm with the wind. The morning sun glints off the skyscrapers as I scan the streets below, my keen nose twitching for any hint of trouble. I nudge my flying partner, Zoom, with my paw, letting out a sharp, excited bark as I point my snout towards a suspicious figure lurking near "The Great Canine Confectionery."


In [147]:
print(performer2.speak())
print(host.decide())

Wayne says: (As Zoom, the flying dog)

*Ears perk up, low growl* Rrruff! I catch the drift of Barker's alert and swoop down beside him, my jet-powered paws leaving a trail of sparkling dog-hair glitter. With a quick series of yips and a head tilt, I signal to Barker that I recognize the shadowy figure - it's the notorious Cat Burglar, known for stealing the city's most prized dog treats and leaving chaos in his wake. My tail starts spinning like a helicopter propeller, ready to give chase and protect our beloved bakery.
continue


In [148]:
for item in Performer.get_full_history():
    print(item)


{'role': 'assistant', 'content': 'Drew says: Hello, I am Drew and I am the host of the Yes-And game. \n    Lets get started!'}
{'role': 'user', 'content': 'What do I do?.'}
{'role': 'assistant', 'content': 'Drew says: Please provide a fun scenario to start the game. It could be anything—like a situation, a location, or a character—just make it creative!'}
{'role': 'user', 'content': 'How about a pair of dogs that can fly and fight crime.'}
{'role': 'assistant', 'content': 'Drew says: Great scenario! Here’s the Scene Brief:\n\n**Setting:** A vibrant city skyline with a mix of tall buildings and parks.  \n**Tone:** Light-hearted and adventurous.  \n**Constraints:** The flying dogs must communicate with each other using dog-like sounds and body language, and they should encounter a mischievous villain trying to steal dog treats.\n\nRyan and Wayne, your scene begins with you as these flying crime-fighting dogs! \n\n[HOST DECISION: Start Game]'}
{'role': 'assistant', 'content': 'Ryan says: 